In [1]:
import pandas as pd
import numpy as np
import h5py
import torch
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import train_test_split
import io

In [2]:
CSV_PATH = '../data/multi-class/insect/metadata_small.parquet'
HDF5_PATH = '../data/multi-class/insect/cropped_256.hdf5'
OUTPUT_CSV = '../data/multi-class/insect/full.parquet'

In [3]:
df = pd.read_parquet(CSV_PATH, engine='pyarrow')

tax_cols = ['phylum', 'class', 'order', 'family']

def join_taxonomy(row):
    # Filtra valores nulos (NaN) e strings vazias
    valid_ranks = [str(val) for val in row if pd.notna(val) and str(val).strip() != '']
    return '.'.join(valid_ranks)

df['taxonomy'] = df[tax_cols].apply(join_taxonomy, axis=1)
df = df.drop(columns=tax_cols)

In [4]:
class_counts = df['taxonomy'].value_counts()
valid_classes = class_counts[class_counts >= 2].index

# Filtramos o dataframe para ter apenas classes estratificáveis
df_stratifiable = df[df['taxonomy'].isin(valid_classes)]

df_50k, _ = train_test_split(
    df_stratifiable, 
    train_size=50000, 
    stratify=df_stratifiable['taxonomy'], 
    random_state=42 # Para reprodutibilidade (sempre os mesmos 50k)
)

df_50k = df_50k.reset_index(drop=True)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
model = torch.nn.Sequential(*(list(model.children())[:-1])) # Remove a última camada (FC)
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Using cache found in /home/gilbert/.cache/torch/hub/pytorch_vision_v0.10.0


In [7]:
embeddings_list = []
indices_to_drop = []

with h5py.File(HDF5_PATH, 'r') as h5f:
    group_name = "bioscan_dataset"
    
    if group_name in h5f.keys():
        hdf5 = h5f[group_name]

    filenames = [k for k in hdf5.keys()]

    for idx, row in df_50k.iterrows():
        print(f'Processing {idx} file...')

        img_name = str(row['image_file'])

        # Verifica se a imagem existe no HDF5
        if img_name not in filenames:
            indices_to_drop.append(idx)
            continue
            
        try:
            # Opção A: Se a imagem estiver salva como bytes (JPEG/PNG) no HDF5
            img_bytes = np.asarray(hdf5[img_name])
            # image = Image.fromarray(img_bytes)
            img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
            
            # Opção B: Se a imagem estiver salva como array NumPy (descomente as linhas abaixo)
            # img_array = np.array(h5f[img_name])
            # img = Image.fromarray(img_array).convert('RGB')
            
            # Aplica transformações e adiciona dimensão de batch
            img_tensor = transform(img).unsqueeze(0).to(device)
            
            # Extrai o embedding
            with torch.no_grad():
                # .squeeze() remove dimensões extras de tamanho 1 (ex: [1, 2048, 1, 1] -> [2048])
                emb = model(img_tensor).squeeze().cpu().numpy()
                embeddings_list.append(emb.tolist()) # Salva como lista para facilitar no CSV
                
        except Exception as e:
            print(f"Erro ao processar a imagem {img_name}: {e}")
            indices_to_drop.append(idx)

# ==========================================
# Passo 4: Atualizar DataFrame e Salvar
# ==========================================
# Dropa as linhas onde as imagens não foram encontradas ou deram erro
df_50k = df_50k.drop(index=indices_to_drop).reset_index(drop=True)

# Adiciona a coluna de embeddings
df_50k['embeddings'] = embeddings_list

# Salva o resultado
df_50k.to_parquet(OUTPUT_CSV, index=False, engine='pyarrow')
print(f"Processamento concluído! Shape final do DataFrame: {df_50k.shape}")

Processing 0 file...
Processing 1 file...
Processing 2 file...
Processing 3 file...
Processing 4 file...
Processing 5 file...
Processing 6 file...
Processing 7 file...
Processing 8 file...
Processing 9 file...
Processing 10 file...
Processing 11 file...
Processing 12 file...
Processing 13 file...
Processing 14 file...
Processing 15 file...
Processing 16 file...
Processing 17 file...
Processing 18 file...
Processing 19 file...
Processing 20 file...
Processing 21 file...
Processing 22 file...
Processing 23 file...
Processing 24 file...
Processing 25 file...
Processing 26 file...
Processing 27 file...
Processing 28 file...
Processing 29 file...
Processing 30 file...
Processing 31 file...
Processing 32 file...
Processing 33 file...
Processing 34 file...
Processing 35 file...
Processing 36 file...
Processing 37 file...
Processing 38 file...
Processing 39 file...
Processing 40 file...
Processing 41 file...
Processing 42 file...
Processing 43 file...
Processing 44 file...
Processing 45 file..